In [1]:
# Step 2: Import Python Packages: Pandas, Numpy, and Statistics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
from scipy.optimize import minimize_scalar
import os

In [2]:
# Step 3: Set I/O Folders
InputPath = 'inputFolder'

os.makedirs('outputFolder', exist_ok=True)
OutputPath = 'outputFolder'

In [3]:
df = pd.read_csv(InputPath + '/CaseStudy_PreTrade.csv')
df.head()

,Number,Trade Date,Symbol,Side,Shares,Price,Volatility,ADV,AlphaBp
0,1,4/21/2025,IBM,Buy,309300,236.22,0.39,5153500,38
1,2,4/21/2025,TRV,Buy,145730,249.59,0.36,1616000,33
2,3,4/21/2025,EMR,Buy,324790,96.42,0.55,3611800,56
3,4,4/21/2025,IFF,Buy,21280,72.71,0.43,2113100,39
4,5,4/21/2025,SO,Buy,50150,90.23,0.23,5008400,22


In [4]:
# Step 4: Set MI Parameters - Calculated from Non-Linear Regression Analysis
a1 = 883.58
a2 = 0.35
a3 = 0.75
a4 = 0.82
b1 = 0.96
MIParams = [a1, a2, a3, a4, b1]

## Helper Functions

### TCA Calculations

In [5]:
# Order Size (%ADV)
def calc_size(shares, adv):
    return shares / adv

# Trade Time (in days)
def calc_time(size, pov):
    return size * (1 - pov) / pov #[cite: 3]

# Market Impact (MI)
def calc_mi(size, vol, pov, a1=883.58, a2=0.35, a3=0.75, a4=0.82, b1=0.96):
    i_star = a1 * (size ** a2) * (vol ** a3) #[cite: 3]
    return i_star * (b1 * (pov ** a4) + (1 - b1)) #[cite: 3]

# Timing Risk (TR)
def calc_tr(vol, trade_time):
    # Annualized volatility scaled to daily volume time
    return vol * np.sqrt((1/3) * (1/250) * trade_time) * 10000 #[cite: 3]

# Price Appreciation (PA)
def calc_pa(side, alpha_bp, trade_time):
    side_sign = 1 if str(side).lower() == 'buy' else -1 #[cite: 3]
    return side_sign * 0.5 * alpha_bp * trade_time #[cite: 3]

### Optimization Calculations

In [6]:
def optimize_traders_dilemma(size, vol, lmbda=0.25):
    # Min: MI + Lambda * TR
    def objective(pov):
        mi = calc_mi(size, vol, pov)
        t_time = calc_time(size, pov)
        tr = calc_tr(vol, t_time)
        return mi + lmbda * tr #[cite: 3]
    
    res = minimize_scalar(objective, bounds=(0.01, 0.99), method='bounded')
    return res.x

def optimize_minimize_cost(size, vol, side, alpha_bp):
    # Min: MI + PA
    def objective(pov):
        mi = calc_mi(size, vol, pov)
        t_time = calc_time(size, pov)
        pa = calc_pa(side, alpha_bp, t_time)
        return mi + pa #[cite: 3]
    
    res = minimize_scalar(objective, bounds=(0.01, 0.99), method='bounded')
    return res.x

def optimize_price_improvement(size, vol, bid=100):
    # Min: (MI - Bid) / TR
    def objective(pov):
        mi = calc_mi(size, vol, pov)
        t_time = calc_time(size, pov)
        tr = calc_tr(vol, t_time)
        return (mi - bid) / tr #[cite: 3]
    
    res = minimize_scalar(objective, bounds=(0.01, 0.99), method='bounded')
    return res.x

## Add 'Size' Column

In [7]:
df['Size'] = calc_size(df['Shares'], df['ADV'])

## POV

In [8]:
base_pov = 0.10
df['Trade_Time'] = calc_time(df['Size'], base_pov)
df['MI'] = calc_mi(df['Size'], df['Volatility'], base_pov)
df['TR'] = calc_tr(df['Volatility'], df['Trade_Time'])
df['PA'] = df.apply(lambda row: calc_pa(row['Side'], row['AlphaBp'], row['Trade_Time']), axis=1)

df.head()

,Number,Trade Date,Symbol,Side,Shares,Price,Volatility,ADV,AlphaBp,Size,Trade_Time,MI,TR,PA
0,1,4/21/2025,IBM,Buy,309300,236.22,0.39,5153500,38,0.060017,0.540157,30.187096,104.663210,10.262986
1,2,4/21/2025,TRV,Buy,145730,249.59,0.36,1616000,33,0.090179,0.811615,32.782362,118.425964,13.391649
2,3,4/21/2025,EMR,Buy,324790,96.42,0.55,3611800,56,0.089925,0.809322,45.004439,180.672806,22.661022
3,4,4/21/2025,IFF,Buy,21280,72.71,0.43,2113100,39,0.010071,0.090635,17.389820,47.269920,1.767375
4,5,4/21/2025,SO,Buy,50150,90.23,0.23,5008400,22,0.010013,0.090119,10.854798,25.211833,0.991305


## Trader's Dilemma - Optimal POV

In [9]:
LMBDA = 0.25
df['Optimal POV'] = df.apply(
    lambda row: optimize_traders_dilemma(row['Size'], row['Volatility'], lmbda=LMBDA), axis=1
)

df.head()

,Number,Trade Date,Symbol,Side,Shares,Price,Volatility,ADV,AlphaBp,Size,Trade_Time,MI,TR,PA,Optimal POV
0,1,4/21/2025,IBM,Buy,309300,236.22,0.39,5153500,38,0.060017,0.540157,30.187096,104.663210,10.262986,0.079650
1,2,4/21/2025,TRV,Buy,145730,249.59,0.36,1616000,33,0.090179,0.811615,32.782362,118.425964,13.391649,0.082257
2,3,4/21/2025,EMR,Buy,324790,96.42,0.55,3611800,56,0.089925,0.809322,45.004439,180.672806,22.661022,0.089366
3,4,4/21/2025,IFF,Buy,21280,72.71,0.43,2113100,39,0.010071,0.090635,17.389820,47.269920,1.767375,0.065870
4,5,4/21/2025,SO,Buy,50150,90.23,0.23,5008400,22,0.010013,0.090119,10.854798,25.211833,0.991305,0.058291


## Minimize Cost

In [10]:
df['Minimize_Cost'] = df.apply(
    lambda row: optimize_minimize_cost(row['Size'], row['Volatility'], row['Side'], row['AlphaBp']), axis=1
)

df.head()

,Number,Trade Date,Symbol,Side,Shares,Price,Volatility,ADV,AlphaBp,Size,Trade_Time,MI,TR,PA,Optimal POV,Minimize_Cost
0,1,4/21/2025,IBM,Buy,309300,236.22,0.39,5153500,38,0.060017,0.540157,30.187096,104.663210,10.262986,0.079650,0.074659
1,2,4/21/2025,TRV,Buy,145730,249.59,0.36,1616000,33,0.090179,0.811615,32.782362,118.425964,13.391649,0.082257,0.082583
2,3,4/21/2025,EMR,Buy,324790,96.42,0.55,3611800,56,0.089925,0.809322,45.004439,180.672806,22.661022,0.089366,0.092640
3,4,4/21/2025,IFF,Buy,21280,72.71,0.43,2113100,39,0.010071,0.090635,17.389820,47.269920,1.767375,0.065870,0.038452
4,5,4/21/2025,SO,Buy,50150,90.23,0.23,5008400,22,0.010013,0.090119,10.854798,25.211833,0.991305,0.058291,0.036260


## Pice Improvement

In [11]:
BID = 100
df['Price_Improvement'] = df.apply(
    lambda row: optimize_price_improvement(row['Size'], row['Volatility'], bid= BID), axis=1
)

df.head()

,Number,Trade Date,Symbol,Side,Shares,Price,Volatility,ADV,AlphaBp,Size,Trade_Time,MI,TR,PA,Optimal POV,Minimize_Cost,Price_Improvement
0,1,4/21/2025,IBM,Buy,309300,236.22,0.39,5153500,38,0.060017,0.540157,30.187096,104.663210,10.262986,0.079650,0.074659,0.190567
1,2,4/21/2025,TRV,Buy,145730,249.59,0.36,1616000,33,0.090179,0.811615,32.782362,118.425964,13.391649,0.082257,0.082583,0.167782
2,3,4/21/2025,EMR,Buy,324790,96.42,0.55,3611800,56,0.089925,0.809322,45.004439,180.672806,22.661022,0.089366,0.092640,0.104447
3,4,4/21/2025,IFF,Buy,21280,72.71,0.43,2113100,39,0.010071,0.090635,17.389820,47.269920,1.767375,0.065870,0.038452,0.556666
4,5,4/21/2025,SO,Buy,50150,90.23,0.23,5008400,22,0.010013,0.090119,10.854798,25.211833,0.991305,0.058291,0.036260,0.989994


## Case to csv

In [12]:
output_file = OutputPath + '/CaseStudy02_Results.csv'
df.to_csv(output_file, index=False)

df.head()

,Number,Trade Date,Symbol,Side,Shares,Price,Volatility,ADV,AlphaBp,Size,Trade_Time,MI,TR,PA,Optimal POV,Minimize_Cost,Price_Improvement
0,1,4/21/2025,IBM,Buy,309300,236.22,0.39,5153500,38,0.060017,0.540157,30.187096,104.663210,10.262986,0.079650,0.074659,0.190567
1,2,4/21/2025,TRV,Buy,145730,249.59,0.36,1616000,33,0.090179,0.811615,32.782362,118.425964,13.391649,0.082257,0.082583,0.167782
2,3,4/21/2025,EMR,Buy,324790,96.42,0.55,3611800,56,0.089925,0.809322,45.004439,180.672806,22.661022,0.089366,0.092640,0.104447
3,4,4/21/2025,IFF,Buy,21280,72.71,0.43,2113100,39,0.010071,0.090635,17.389820,47.269920,1.767375,0.065870,0.038452,0.556666
4,5,4/21/2025,SO,Buy,50150,90.23,0.23,5008400,22,0.010013,0.090119,10.854798,25.211833,0.991305,0.058291,0.036260,0.989994
